# Connecting the Python AIoT Device to MQTT — Lecture 4

We built the virtual device in Lectures 2 and 3. Now we connect it to an MQTT broker and observe its telemetry through a separate subscriber.

**Python AIoT Device → MQTT Broker → MQTT Subscriber**

MQTT fundamentals were covered earlier in the course; this notebook focuses on practical integration.


## 1. Install Paho MQTT

Run in a terminal:

```bash
pip install paho-mqtt
```

The Eclipse Paho Python client provides MQTT connection, publish, subscribe and network-loop capabilities.


In [ ]:
import paho.mqtt
print("Paho MQTT imported successfully.")
print("Version:", getattr(paho.mqtt, "__version__", "unknown"))


## 2. Configure the Device and Topic

In [ ]:
BROKER_HOST = "localhost"
BROKER_PORT = 1883
DEVICE_ID = "MCH-007"
MQTT_TOPIC = "factory/production-line-2/machine/MCH-007/telemetry"

print("Broker:", f"{BROKER_HOST}:{BROKER_PORT}")
print("Topic :", MQTT_TOPIC)


## 3. Connect the Python MQTT Client

Paho needs a client, a broker connection, and a network loop to process MQTT traffic.


In [ ]:
import time
import paho.mqtt.client as mqtt

def on_connect(client, userdata, flags, reason_code, properties):
    if reason_code.is_failure:
        print("Connection failed:", reason_code)
    else:
        print("Connected to MQTT broker successfully.")

client = mqtt.Client(
    mqtt.CallbackAPIVersion.VERSION2,
    client_id=DEVICE_ID,
)
client.on_connect = on_connect
client.connect(BROKER_HOST, BROKER_PORT, 60)
client.loop_start()
time.sleep(1)


## 4. Create a Telemetry Message

In [ ]:
import json
from datetime import datetime
import random

telemetry = {
    "device_id": DEVICE_ID,
    "device_type": "Bottle Filling Machine",
    "location": "Production Line 2",
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "state": "RUNNING",
    "message": "Normal operation",
    "readings": {
        "temperature": round(random.uniform(70, 75), 2),
        "vibration": round(random.uniform(1.8, 2.4), 2),
        "pressure": round(random.uniform(6.0, 6.5), 2),
        "energy": round(random.uniform(1.2, 1.8), 2),
    },
}

print(json.dumps(telemetry, indent=2))


## 5. Publish the Telemetry

The critical integration is:

**Telemetry → JSON → `publish()` → MQTT Broker**


In [ ]:
payload = json.dumps(telemetry)

info = client.publish(
    MQTT_TOPIC,
    payload=payload,
    qos=0,
    retain=False,
)

print("Publish result code:", info.rc)
print("Published to:", MQTT_TOPIC)


## 6. Live End-to-End Demonstration

Open a second terminal and run:

```bash
python subscriber.py
```

Then, in another terminal:

```bash
python device_mqtt.py
```

The subscriber should receive the device telemetry in real time.

You should now be able to see:

**Python Device → MQTT Broker → Subscriber**


## 🧪 Learner Challenge — Change the Topic

Try:

```text
factory/production-line-2/machine/MCH-007/status
```

Then consider: would this be the best topic for continuous sensor telemetry? Why or why not?


# What We Have Achieved

Our virtual AIoT device is now connected.

It can:
- Connect to an MQTT broker
- Publish structured telemetry
- Use a meaningful topic
- Send data to an independent subscriber

### Next

So far communication is one-way:

**Device → Enterprise**

Next we will implement:

**Enterprise → Device**

with device commands and two-way MQTT communication.


In [ ]:
# Clean up the notebook connection
client.loop_stop()
client.disconnect()
print("Disconnected cleanly.")
